In [ ]:
import os

In [ ]:
def parse_ravdess_filename(path: str) -> dict:
    filename = os.path.basename(path)
    audiofilepart = filename.replace(".wav", "").split("-")

    modality_map = {
        "01": "audio-only",
        "02": "audio-video"
    }

    vocal_map = {
        "01": "speech",
        "02": "song"
    }

    emotion_map = {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fearful",
        "07": "disgust",
        "08": "surprised"
    }

    intensity_map = {
        "01": "normal",
        "02": "strong"
    }

    statement_map = {
        "01": "kids are talking",
        "02": "dogs are sitting"
    }

    repetition_map = {
        "01": "first",
        "02": "second"
    }

    return {
        "modality": modality_map.get(audiofilepart[0]),
        "vocal_channel": vocal_map.get(audiofilepart[1]),
        "emotion": emotion_map.get(audiofilepart[2]),
        "intensity": intensity_map.get(audiofilepart[3]),
        "statement": statement_map.get(audiofilepart[4]),
        "repetition": repetition_map.get(audiofilepart[5]),
        "actor": f"Actor_{audiofilepart[6]}"
    }

In [ ]:
DATASET_PATH = "data/ravdess"

In [ ]:
import os

print(os.listdir("data"))
print(os.listdir("data/ravdess")[:5])

In [ ]:
for root, _, files in os.walk(DATASET_PATH):
    for f in files:
        if f.endswith(".wav"):
            info = parse_ravdess_filename(f)
            print(info)
            break
    break

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import numpy as np
import librosa.display

In [ ]:
DATASET_PATH = "data/ravdess"


In [ ]:
rows = []

for root, _, files in os.walk(DATASET_PATH):
    for f in files:
        if f.endswith(".wav"):
            info = parse_ravdess_filename(f)

            # Only speech subset
            if info["vocal_channel"] == "speech":
                rows.append(info["emotion"])

df = pd.DataFrame(rows, columns=["emotion"])

counts = df["emotion"].value_counts()

counts

In [ ]:
counts.plot(kind="bar")
plt.title("Emotion Distribution (Speech Only)")
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.show()


In [ ]:
durations = []

for root, _, files in os.walk(DATASET_PATH):
    for f in files:
        if f.endswith(".wav"):
            path = os.path.join(root, f)

            y, sr = librosa.load(path)
            duration = len(y) / sr
            durations.append(duration)

durations = np.array(durations)

print("Mean duration:", durations.mean())
print("Standard deviation:", durations.std())

In [ ]:
plt.hist(durations, bins=20)
plt.title("Distribution of Audio Durations")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of Files")
plt.show()

In [ ]:
samples = {"happy": None, "sad": None, "angry": None}

for root, _, files in os.walk(DATASET_PATH):
    for f in files:
        if f.endswith(".wav"):
            info = parse_ravdess_filename(f)

            if info["emotion"] in samples and samples[info["emotion"]] is None:
                samples[info["emotion"]] = os.path.join(root, f)

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(12, 10))

for i, (emotion, path) in enumerate(samples.items()):
    y, sr = librosa.load(path)

    # Waveform
    ax[i, 0].plot(y)
    ax[i, 0].set_title(f"{emotion} - Waveform")

    # Mel Spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr)
    S_db = librosa.power_to_db(S)

    librosa.display.specshow(S_db, sr=sr, ax=ax[i, 1])
    ax[i, 1].set_title(f"{emotion} - Mel Spectrogram")

plt.tight_layout()
plt.show()

In [ ]:
import opensmile
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)


In [ ]:
file_path = r"data\ravdess\Actor_01\03-02-01-01-01-01-01.wav"

In [ ]:
features = smile.process_file(file_path)

In [ ]:
print(os.path.exists(file_path))

In [ ]:
print(features.shape)

In [ ]:
print(features.columns.tolist())

In [ ]:
print(features.values)

In [ ]:
import os
import pandas as pd
import opensmile

In [ ]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)


In [ ]:
DATASET_PATH = "data/ravdess"

In [ ]:
from tqdm import tqdm

rows = []

for root, _, files in os.walk(DATASET_PATH):
    for f in tqdm(files):
        if f.endswith(".wav"):
            path = os.path.join(root, f)
            
            feat = smile.process_file(path)
            emotion = f.split("-")[2]
            feat["label"] = emotion
            
            rows.append(feat)


In [ ]:
df = pd.concat(rows).reset_index(drop=True)

In [ ]:
print(df.shape)


In [ ]:
df.to_csv("ravdess_egemaps.csv", index=False)
print("Saved 100%")



In [ ]:
df = pd.read_csv("ravdess_egemaps.csv")


In [ ]:
pitch_feature = "F0semitoneFrom27.5Hz_sma3nz_amean"

mean_pitch = df.groupby("label")[pitch_feature].mean()

mean_pitch

In [ ]:
mean_pitch.sort_values().plot(kind="barh")
plt.title("Mean Pitch per Emotion")
plt.xlabel("Pitch (mean)")
plt.ylabel("Emotion")
plt.show()

In [ ]:
corr = df.drop("label", axis=1).corr()


In [ ]:

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()


In [ ]:

corr_pairs = corr.abs().unstack()

# Remove self-correlations
corr_pairs = corr_pairs[corr_pairs < 1]

# Select strong ones
high_corr = corr_pairs[corr_pairs > 0.9]

high_corr.sort_values(ascending=False).head(10)